In [ ]:
from Classes.Grid import Grid
from Classes.ScalarCoeffs import ScalarCoeffs
from Classes.BoundaryConditions import BoundaryLocation, DirichletBc, NeumannBc, RobinBc
from Classes.Models import DiffusionModel, SurfaceConvectionModel
from Classes.LinearSolver import solve

import numpy as np
from numpy.linalg import norm
import matplotlib.pyplot as plt

## First order model

In [ ]:
class FirstOrderTransientModel:
    """Class defining a first order implicit transient model"""

    def __init__(self, grid, T, Told, rho, cp, dt):
        """Constructor"""
        self._grid = grid
        self._T = T
        self._Told = Told
        self._rho = rho
        self._cp = cp
        self._dt = dt

    def add(self, coeffs):
        """Function to add transient term to coefficient arrays"""

        # Calculate the transient term
        rp_coeff = (self._rho * self._cp * self._grid.vol/ self._dt) * (self._T[1:-1] - self._Told[1:-1])

        # Calculate the linearization coefficient
        ap_coeff = (self._rho * self._cp * self._grid.vol/ dt)
        
        # Add to coefficient arrays
        coeffs.accumulate_aP(ap_coeff)
        coeffs.accumulate_rP(rp_coeff)
        
        return coeffs

## Second order model

In [ ]:
class SecondOrderTransientModel:
    """Class defining a second order implicit transient model"""

    def __init__(self, grid, T, Told, Told2, rho, cp, dt):
        """Constructor"""
        self._grid = grid
        self._T = T
        self._Told = Told
        self._Told2 = Told2
        self._rho = rho
        self._cp = cp
        self._dt = dt

    def add(self, coeffs):
        """Function to add transient term to coefficient arrays"""
        
        # Calculate the transient term rhp * cp * V *(3T - 4Told + T^Told2) / (2 dt)
        rp_coeff = self._rho * self._cp * self._grid.vol * (3.0 * self._T[1:-1] - 4.0 * self._Told[1:-1] + self._Told2[1:-1]) / (2.0 * self._dt)

        # Calculate the linearization coefficient 3 rho cp V / (2 dt)
        ap_coeff = 3.0 * self._rho * self._cp * self._grid.vol / (2.0 * self._dt)
        
        # Add to coefficient arrays
        coeffs.accumulate_aP(ap_coeff)
        coeffs.accumulate_rP(rp_coeff)
        
        return coeffs

## Analytical Function

In [ ]:
#define funciton for analytical solution
def analytical_T(x, tau):
    return To + (Ti - To) * C1 * np.exp(-zeta**2 * tau) * np.cos(zeta * x)

## Parameters define

In [ ]:
#Parameters
Bi = 1
    
Ti = 100 #C
To = 0    #C
    
C1 = 1.1191
zeta = 0.8603
    
tau1 = 0.4535
tau2 = 3.2632
    
    
    
# Define the grid
lx = 1.0
ly = 0.1
lz = 0.1

# Set the timestep information
time = 0
 
# Set the maximum number of iterations and convergence criterion
maxIter = 10
converged = 1e-6
    
# Define thermophysical properties
rho = 1
cp = 1
k = 1
h = Bi * k / lx


## Define funciton for loop-running

In [ ]:
#define function for running first order
def runcase_firstorder(grid, ntime):    
    
    # define local dt and grid
    dt = (tau2 - tau1) / ntime

    # Define the coefficients
    coeffs = ScalarCoeffs(grid.ncv)
    
    # Initial conditions
    #T0 = 300
    
    # Initialize field variable arrays
    T = analytical_T(grid.xP, tau1)
    
    # Define boundary conditions
    west_bc = NeumannBc(T, grid, 0.0, BoundaryLocation.WEST)
    east_bc = RobinBc(T, grid, h, k, To, BoundaryLocation.EAST)
    
    # Apply boundary conditions
    west_bc.apply()
    east_bc.apply()
    
    # Create list to store the solutions at each timestep
    # Note: If there are a lot of timesteps, this will cost a
    #       lot of memory. It is suggested to set a parameter to 
    #       only store the solution every N timesteps.
    T_solns = [np.copy(T)]
    
    # Define the transient model
    Told = np.copy(T)
    
    transient = FirstOrderTransientModel(grid, T, Told, rho, cp, dt)
    
    # Define the diffusion model
    diffusion = DiffusionModel(grid, T, k, west_bc, east_bc)
    
    time = tau1
    
    # Define the surface convection model (no need)
    #surfaceConvection = SurfaceConvectionModel(grid, T, ho, To)
    
    # Loop through all timesteps
    
    for tStep in range(ntime):
        # Update the time information
        time += dt
        
        # Print the timestep information
        print("First order: Timestep = {}; Time = {}".format(tStep, time))
        
        # Store the "old" temperature field
        # Note: do not use copy here because that creates a new object
        #       and we want the reference in the transient model to remain
        #       valid
        Told[:] = T[:]
        
        # Iterate until the solution is converged
        for i in range(maxIter):
            # Zero the coefficients and add each influence
            coeffs.zero()
            coeffs = diffusion.add(coeffs)
            #coeffs = surfaceConvection.add(coeffs)
            coeffs = transient.add(coeffs)
            
            # Compute residual and check for convergence 
            maxResid = norm(coeffs.rP, np.inf)
            avgResid = np.mean(np.absolute(coeffs.rP))
            print("First order: Iteration = {}; Max. Resid. = {}; Avg. Resid. = {}".format(i, maxResid, avgResid))
            
            if maxResid < converged:
                break
        
            # Solve the sparse matrix system
            dT = solve(coeffs)
        
            # Update the solution and boundary conditions
            T[1:-1] += dT
            west_bc.apply()
            east_bc.apply()
        
        # Store the solution
        T_solns.append(np.copy(T))

    return T, dt

In [ ]:
#define function for running second order
def runcase_secondorder(grid, ntime):
    
    # define local dt and grid
    dt = (tau2 - tau1) / ntime
    
    # Define the coefficients
    coeffs = ScalarCoeffs(grid.ncv)
    
    # Initial conditions
    #T0 = 300
    
    # Initialize field variable arrays
    T = analytical_T(grid.xP, tau1)
    
    # Define boundary conditions
    west_bc = NeumannBc(T, grid, 0.0, BoundaryLocation.WEST)
    east_bc = RobinBc(T, grid, h, k, To, BoundaryLocation.EAST)
    
    # Apply boundary conditions
    west_bc.apply()
    east_bc.apply()
    
    # Create list to store the solutions at each timestep
    T_solns = [np.copy(T)]
    
    # Define the surface convection model
    #surfaceConvection = SurfaceConvectionModel(grid, T, ho, To)
    
    # Loop through all timesteps
    Told = np.copy(T)
    Told2 = np.copy(T)
    
    diffusion = DiffusionModel(grid, T, k, west_bc, east_bc)
    
    time = tau1
    
    for tStep in range(ntime):
    
        time += dt
    
        print("Second order: Timestep = {}; Time = {}".format(tStep + 1, time))
    
        if tStep == 0:
            # First timestep: use first-order implicit because T^{n-1} is unavailable
            Told[:] = T[:]
    
            transient = FirstOrderTransientModel(
                grid, T, Told, rho, cp, dt
            )
    
        else:
            # Second-order implicit:
            # Told  = T^n
            # Told2 = T^{n-1}
            Told2[:] = Told[:]
            Told[:] = T[:]
    
            transient = SecondOrderTransientModel(
                grid, T, Told, Told2, rho, cp, dt
            )
    
        for i in range(maxIter):
    
            coeffs.zero()
    
            coeffs = diffusion.add(coeffs)
            coeffs = transient.add(coeffs)
    
            maxResid = norm(coeffs.rP, np.inf)
            avgResid = np.mean(np.absolute(coeffs.rP))
    
            print("Second order: Iteration = {}; Max. Resid. = {}; Avg. Resid. = {}".format(
                i, maxResid, avgResid
            ))
    
            if maxResid < converged:
                break
    
            dT = solve(coeffs)
    
            T[1:-1] += dT
    
            west_bc.apply()
            east_bc.apply()
    
        T_solns.append(np.copy(T))

    return T, dt

In [ ]:
#running as a loop for differet time steps
nTime_list = np.array([2, 4, 8, 16, 32])

ncv = 5
grid = Grid(lx, ly, lz, ncv)

#define array for error and T0

dt_values = []

error_first = []
error_second = []

T0_first = []
T0_second = []

T_exact_final = analytical_T(grid.xP, tau2)


for nTime in nTime_list:

    
    dt = (tau2 - tau1) / nTime
    
    print(f"Running nTime = {nTime}")
    print(f"Running dt = {dt}")
    
    T_first, dt = runcase_firstorder(grid, nTime)
    T_second, dt = runcase_secondorder(grid, nTime)
    
    
    dt_values.append(dt)

    # Average absolute error over interior CVs
    err_first = np.mean(np.abs(T_exact_final[1:-1] - T_first[1:-1]))
    err_second = np.mean(np.abs(T_exact_final[1:-1] - T_second[1:-1]))

    error_first.append(err_first)
    error_second.append(err_second)

    # T(0, t2)
    T0_first.append(T_first[0])
    T0_second.append(T_second[0])

dt_values = np.array(dt_values)

error_first = np.array(error_first)
error_second = np.array(error_second)

T0_first = np.array(T0_first)
T0_second = np.array(T0_second)


## Independence test

In [ ]:
#check mesh independent
print("5 ncv --> First order: T final at ntime = 2 --> {}".format(T0_first[1]))
print("5 ncv --> Second order: T final at ntime = 2 --> {}".format(T0_second[1]))


In [ ]:
#rerun for 10 ncv
#running as a loop for differet time steps
ncv = 10  # increase from 5 to 10
grid = Grid(lx, ly, lz, ncv)

#define array for error and T0

dt_values = []

error_first = []
error_second = []

T0_first = []
T0_second = []

T_exact_final = analytical_T(grid.xP, tau2)


for nTime in nTime_list:

    
    dt = (tau2 - tau1) / nTime
    
    print(f"Running nTime = {nTime}")
    print(f"Running dt = {dt}")
    
    T_first, dt = runcase_firstorder(grid, nTime)
    T_second, dt = runcase_secondorder(grid, nTime)
    
    
    dt_values.append(dt)

    # Average absolute error over interior CVs
    err_first = np.mean(np.abs(T_exact_final[1:-1] - T_first[1:-1]))
    err_second = np.mean(np.abs(T_exact_final[1:-1] - T_second[1:-1]))

    error_first.append(err_first)
    error_second.append(err_second)

    # T(0, t2)
    T0_first.append(T_first[0])
    T0_second.append(T_second[0])

dt_values = np.array(dt_values)

error_first = np.array(error_first)
error_second = np.array(error_second)

T0_first = np.array(T0_first)
T0_second = np.array(T0_second)


In [ ]:
#check mesh independent
print("10 ncv --> First order: T final at ntime = 2 --> {}".format(T0_first[1]))
print("10 ncv --> Second order: T final at ntime = 2 --> {}".format(T0_second[1]))


In [ ]:
#rerun for 20 ncv
#running as a loop for differet time steps
ncv = 20
grid = Grid(lx, ly, lz, ncv)

#define array for error and T0

dt_values = []

error_first = []
error_second = []

T0_first = []
T0_second = []

T_exact_final = analytical_T(grid.xP, tau2)


for nTime in nTime_list:

    
    dt = (tau2 - tau1) / nTime
    
    print(f"Running nTime = {nTime}")
    print(f"Running dt = {dt}")
    
    T_first, dt = runcase_firstorder(grid, nTime)
    T_second, dt = runcase_secondorder(grid, nTime)
    
    
    dt_values.append(dt)

    # Average absolute error over interior CVs
    err_first = np.mean(np.abs(T_exact_final[1:-1] - T_first[1:-1]))
    err_second = np.mean(np.abs(T_exact_final[1:-1] - T_second[1:-1]))

    error_first.append(err_first)
    error_second.append(err_second)

    # T(0, t2)
    T0_first.append(T_first[0])
    T0_second.append(T_second[0])

dt_values = np.array(dt_values)

error_first = np.array(error_first)
error_second = np.array(error_second)

T0_first = np.array(T0_first)
T0_second = np.array(T0_second)

In [ ]:
#check mesh independent
print("20 ncv --> First order: T final at ntime = 2 --> {}".format(T0_first[1]))
print("20 ncv --> Second order: T final at ntime = 2 --> {}".format(T0_second[1]))


## Mesh Independence
A grid with 10 control volumes was selected because further mesh refinement produced less than a 5% change in the final temperature. Therefore, the solution was considered mesh independent for the present study.

In [ ]:
## run as ncv = 10
nTime_list = np.array([2, 4, 8, 16, 32])

ncv = 10
grid = Grid(lx, ly, lz, ncv)

#define array for error and T0

dt_values = []

error_first = []
error_second = []

T0_first = []
T0_second = []

T_exact_final = analytical_T(grid.xP, tau2)


for nTime in nTime_list:

    
    dt = (tau2 - tau1) / nTime
    
    print(f"Running nTime = {nTime}")
    print(f"Running dt = {dt}")
    
    T_first, dt = runcase_firstorder(grid, nTime)
    T_second, dt = runcase_secondorder(grid, nTime)
    
    
    dt_values.append(dt)

    # Average absolute error over interior CVs
    err_first = np.mean(np.abs(T_exact_final[1:-1] - T_first[1:-1]))
    err_second = np.mean(np.abs(T_exact_final[1:-1] - T_second[1:-1]))

    error_first.append(err_first)
    error_second.append(err_second)

    # T(0, t2)
    T0_first.append(T_first[0])
    T0_second.append(T_second[0])

dt_values = np.array(dt_values)

error_first = np.array(error_first)
error_second = np.array(error_second)

T0_first = np.array(T0_first)
T0_second = np.array(T0_second)

## Plot T profile

In [ ]:
%matplotlib inline
plt.figure(figsize=(7, 5))

plt.plot(
    grid.xP[1:-1],
    T_first[1:-1],
    "o-",
    linewidth=2,
    label="First-order implicit"
)

plt.plot(
    grid.xP[1:-1],
    T_second[1:-1],
    "s-",
    linewidth=2,
    label="Second-order implicit"
)

plt.plot(
    grid.xP[1:-1],
    T_exact_final[1:-1],
    "k--",
    linewidth=2,
    label="Exact"
)

plt.xlabel("x")
plt.ylabel(r"$T(x,t_2)$")
plt.grid(True)
plt.legend()
plt.show()

## Plot error vs time step and P value

In [ ]:
%matplotlib inline

p_first, logc_first = np.polyfit(np.log(dt_values), np.log(error_first), 1)
p_second, logc_second = np.polyfit(np.log(dt_values), np.log(error_second), 1)

print(f"First-order implicit p  = {p_first:.4f}")
print(f"Second-order implicit p = {p_second:.4f}")

plt.figure(figsize=(7, 5))

plt.loglog(
    dt_values,
    error_first,
    "o-",
    linewidth=2,
    label=f"First-order implicit, p = {p_first:.2f}"
)

plt.loglog(
    dt_values,
    error_second,
    "s-",
    linewidth=2,
    label=f"Second-order implicit, p = {p_second:.2f}"
)

plt.xlabel(r"$\Delta \tau$")
plt.ylabel(r"Average absolute error, $\overline{e}$")
plt.grid(True, which="both")
plt.legend()
plt.show()


## Plot T(0) at final time

In [ ]:
%matplotlib inline
plt.figure(figsize=(7, 5))

plt.plot(
    nTime_list,
    T0_first,
    "o-",
    linewidth=2,
    label="First-order implicit"
)

plt.plot(
    nTime_list,
    T0_second,
    "s-",
    linewidth=2,
    label="Second-order implicit"
)

plt.axhline(
    y=T_exact_final[0],
    linestyle="--",
    linewidth=2,
    label="Exact"
)

plt.xlabel("Number of timesteps")
plt.ylabel(r"$T(0,t_2)$")
plt.grid(True)
plt.legend()
plt.show()

## The second-order implicit scheme is more accurate than the first-order implicit scheme when compared with the analytical solution. It also shows faster temporal convergence as the number of time steps increases.

## Crank–Nicolson scheme

In [ ]:
class DiffusionModel_crank:
    """Class defining a diffusion model for crank nicolson"""
    
    def __init__(self, grid, phi, gamma, west_bc, east_bc):
        """Constructor"""
        self._grid = grid
        self._phi = phi
        self._gamma = gamma
        self._west_bc = west_bc
        self._east_bc = east_bc
        
    def add(self, coeffs):
        """Function to add diffusion terms to coefficient arrays"""
        
        # Calculate the west and east face diffusion flux terms for each face
        flux_w = - self._gamma*self._grid.Aw*(self._phi[1:-1]-self._phi[0:-2])/self._grid.dx_WP
        flux_e = - self._gamma*self._grid.Ae*(self._phi[2:]-self._phi[1:-1])/self._grid.dx_PE
        
        # Calculate the linearization coefficients
        coeffW = - self._gamma*self._grid.Aw/self._grid.dx_WP
        coeffE = - self._gamma*self._grid.Ae/self._grid.dx_PE
        coeffP = - coeffW - coeffE
        
        # Modify the linearization coefficients on the boundaries
        coeffP[0] += coeffW[0]*self._west_bc.coeff()
        coeffP[-1] += coeffE[-1]*self._east_bc.coeff()
        
        # Zero the boundary coefficients that are not used
        coeffW[0] = 0.0
        coeffE[-1] = 0.0
        
        # Calculate the net flux from each cell
        flux = flux_e - flux_w
        
        # Add to coefficient arrays 0.5 weight added
        
        coeffs.accumulate_aP(0.5 * coeffP)
        coeffs.accumulate_aW(0.5 * coeffW)
        coeffs.accumulate_aE(0.5 * coeffE)
        coeffs.accumulate_rP(0.5 * flux)
        
        # Return the modified coefficient array
        return coeffs
        

In [ ]:
#define function for running first order with crank ni
def runcase_firstorder_crank(grid, ntime):    
    
    # define local dt and grid
    dt = (tau2 - tau1) / ntime

    # Define the coefficients
    coeffs = ScalarCoeffs(grid.ncv)
    
    # Initial conditions
    #T0 = 300
    
    # Initialize field variable arrays
    T = analytical_T(grid.xP, tau1)
    
    # Define boundary conditions
    west_bc = NeumannBc(T, grid, 0.0, BoundaryLocation.WEST)
    east_bc = RobinBc(T, grid, h, k, To, BoundaryLocation.EAST)
    
    # Apply boundary conditions
    west_bc.apply()
    east_bc.apply()

    # Old temperature field
    Told = np.copy(T)

    # Boundary conditions for old solution
    west_bc_old = NeumannBc(Told, grid, 0.0, BoundaryLocation.WEST)
    east_bc_old = RobinBc(Told, grid, h, k, To, BoundaryLocation.EAST)

    west_bc_old.apply()
    east_bc_old.apply()
    
    # Create list to store the solutions at each timestep
    # Note: If there are a lot of timesteps, this will cost a
    #       lot of memory. It is suggested to set a parameter to 
    #       only store the solution every N timesteps.
    T_solns = [np.copy(T)]

    
    # Define the transient model
    
    transient = FirstOrderTransientModel(grid, T, Told, rho, cp, dt)
    
    # Define the diffusion model
    diffusion = DiffusionModel_crank(grid, T, k, west_bc, east_bc)

    diffusion_old = DiffusionModel_crank(grid, Told, k, west_bc_old, east_bc_old)
    
    time = tau1
    
    # Define the surface convection model (no need)
    #surfaceConvection = SurfaceConvectionModel(grid, T, ho, To)
    
    # Loop through all timesteps
    
    for tStep in range(ntime):

        time += dt
        print("Crank-Nicolson: Timestep = {}; Time = {}".format(tStep, time))
    
        # Store old temperature
        Told[:] = T[:]
    
        # Apply old boundary conditions
        west_bc_old.apply()
        east_bc_old.apply()
    
        # Compute 0.5 * old diffusion residual once per timestep
        old_coeffs = ScalarCoeffs(grid.ncv)
        old_coeffs.zero()
        old_coeffs = diffusion_old.add(old_coeffs)
    
        rPold_diffusion = np.copy(old_coeffs.rP)
    
        for i in range(maxIter):
    
            coeffs.zero()
    
            # Adds 0.5 * new diffusion
            coeffs = diffusion.add(coeffs)
    
            # Adds 0.5 * old diffusion
            coeffs.accumulate_rP(rPold_diffusion)
    
            # Adds transient term
            coeffs = transient.add(coeffs)
    
            maxResid = norm(coeffs.rP, np.inf)
            avgResid = np.mean(np.absolute(coeffs.rP))
    
            print(
                "Crank-Nicolson: Iteration = {}; Max. Resid. = {}; Avg. Resid. = {}"
                .format(i, maxResid, avgResid)
            )
    
            if maxResid < converged:
                break
    
            dT = solve(coeffs)
    
            T[1:-1] += dT
    
            west_bc.apply()
            east_bc.apply()

    return T, dt

In [ ]:
## run as ncv = 10

ncv = 10
grid = Grid(lx, ly, lz, ncv)

#define array for error and T0

dt_values = []

error_first_crank = []


T0_first_crank = []


T_exact_final = analytical_T(grid.xP, tau2)


nTime = 32

    
dt = (tau2 - tau1) / nTime
    
print(f"Running nTime = {nTime}")
print(f"Running dt = {dt}")
    
T_first_Crank, dt = runcase_firstorder_crank(grid, nTime)

    
    
dt_values.append(dt)

# Average absolute error over interior CVs
err_first_crank = np.mean(np.abs(T_exact_final[1:-1] - T_first_Crank[1:-1]))


error_first_crank.append(err_first)


# T(0, t2)
T0_first_crank.append(T_first_Crank[0])


dt_values = np.array(dt_values)

error_first_crank = np.array(error_first_crank)


T0_first_crank = np.array(T0_first_crank)


In [ ]:
%matplotlib inline
plt.figure(figsize=(7, 5))

plt.plot(grid.xP, 
    T_first_Crank,
    "o-",
    linewidth=2,
    label="Crank-Nicolson"
)
plt.plot(grid.xP, 
    T_exact_final,
    "o-",
    linewidth=2,
    label="Analytical solution"
)

plt.xlabel("x")
plt.ylabel(r"T")
plt.grid(True)
plt.legend()
plt.show()